In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

import numpy as np

import os
import sys

sys.path.append(os.path.abspath(".."))

from src.metrics import evaluate_model_metrics
from src.dataloader import EEG_Dataset
from model.CLEnet import CLEnet

ModuleNotFoundError: No module named 'torch'

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current device: {device}")

# Links

### [CLEnet article](https://www.nature.com/articles/s41598-025-98653-1)
### [EMA-1D](https://arxiv.org/pdf/2305.13563)
#### [Github](https://github.com/YOLOonMe/EMA-attention-module)

# Dataloader & Dataset

In [ ]:
# ===== Training =====
train_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme", # path to training dir
    "training_epochs"
)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)


# ===== Valid =====
valid_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme", # path to validation dir
    "validation_epochs"
)
valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=64, shuffle=False)


# ===== Test =====
test_dataset = EEG_Dataset(
    "/kaggle/input/eeg-clean-raw/dat-dataset-2-pro-max-supreme", # path to testing dir
    "testing_epochs"
)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

# Training

In [ ]:
# ===== Hyperparameters =====
channel_plan = [1, 16, 32, 128]
kernel_sizes = [3, 7]
stride = 1
padding = 1
factor = 8
kernel_pool = 2
stride_pool = 2
pool_dropout = 'pool'
cnn_plan = [128, 256, 512]
num_layers = 1
in_features = 59392
out_features = 512
learning_rate = 1e-4


# ===== Model, Loss, Optimizer =====
model = CLEnet(
    channel_plan=channel_plan,
    kernel_sizes=kernel_sizes,
    stride=stride,
    padding=padding,
    factor=factor,
    kernel_pool=kernel_pool,
    stride_pool=stride_pool,
    pool_dropout=pool_dropout,
    cnn_plan=cnn_plan,
    num_layers=num_layers,
    in_features=in_features,
    out_features=out_features
)
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.05, 0.9))

num_epochs = 20
train_loss_history = []
val_loss_history = []

# ===== Paths for saving =====
checkpoint_dir = '/kaggle/working/models' # Change to desired file path
os.makedirs(checkpoint_dir, exist_ok=True)
checkpoint_path = os.path.join(checkpoint_dir, 'clenet_checkpoint.pth')

# ===== Resume if model checkpoint exists =====
start_epoch = 0
train_loss_history = []
val_loss_history = []

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_loss_history = checkpoint['train_loss_history']
    val_loss_history = checkpoint['val_loss_history']
    print(f"Resumed from epoch {start_epoch}")


# ===== Early Stopping Parameters =====
best_val_loss = float('inf')
epochs_no_improve = 0
best_model_state = None
patience = 20

# ===== Training =====
for epoch in range(start_epoch, num_epochs):
    model.train()
    train_running_loss = 0.0

    for batch_idx, (raw, clean) in enumerate(train_loader):
        raw = raw.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()
        outputs = model(raw)
        loss = criterion(outputs, clean)
        loss.backward()
        optimizer.step()

        train_running_loss += loss.item()

    train_avg_loss = train_running_loss / len(train_loader)
    train_loss_history.append(train_avg_loss)

    # ====== Validation ======
    model.eval()
    val_running_loss = 0.0

    with torch.no_grad():
        for val_raw, val_clean in valid_loader:
            val_raw = val_raw.to(device)
            val_clean = val_clean.to(device)

            val_outputs = model(val_raw)
            val_loss = criterion(val_outputs, val_clean)
            val_running_loss += val_loss.item()

    val_avg_loss = val_running_loss / len(valid_loader)
    val_loss_history.append(val_avg_loss)

    # ===== Early Stopping Check =====
    if val_avg_loss < best_val_loss:
        best_val_loss = val_avg_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict()
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_avg_loss:.6f} | Val Loss: {val_avg_loss:.6f} | No Improve: {epochs_no_improve}")

    if epochs_no_improve >= patience:
        print(f"Early stopping after {epoch+1} epochs.")
        break
        
    # Save checkpoint every 5 epochs
    if (epoch + 1) % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss_history': train_loss_history,
            'val_loss_history': val_loss_history,
            'hyperparameters': {
                'channel_plan': channel_plan,
                'kernel_sizes': kernel_sizes,
                'stride': stride,
                'padding': padding,
                'factor': factor,
                'kernel_pool': kernel_pool,
                'stride_pool': stride_pool,
                'pool_dropout': pool_dropout,
                'cnn_plan': cnn_plan,
                'num_layers': num_layers,
                'in_features': in_features,
                'out_features': out_features,
                'learning_rate': learning_rate,
                'num_epochs': num_epochs
            }
        }, checkpoint_path)
        print(f"Checkpoint saved to: {checkpoint_path}")



# ===== Saving model =====
model_dir = '/kaggle/working/models' # Change to desired file path
os.makedirs(model_dir, exist_ok=True)
model_cpu = model.to('cpu') # Switching back to cpu

# Save the model state dictionary
model_path = os.path.join(model_dir, 'clenet.pth')

# Save training history and hyperparameters
torch.save({
    'train_loss_history': train_loss_history,
    'val_loss_history': val_loss_history,
    'hyperparameters': {
        'channel_plan': channel_plan,
        'kernel_sizes': kernel_sizes,
        'stride': stride,
        'padding': padding,
        'factor': factor,
        'kernel_pool': kernel_pool,
        'stride_pool': stride_pool,
        'pool_dropout': pool_dropout,
        'cnn_plan': cnn_plan,
        'num_layers': num_layers,
        'in_features': in_features,
        'out_features': out_features,
        'learning_rate': learning_rate,
        'num_epochs': num_epochs
    }
}, model_path)
print(f"Training history saved to: {model_path}")

# Testing

In [ ]:
# Printing metrics
metrics = evaluate_model_metrics(model, test_loader)

print("Test metrics:")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")
    
with open('metrics_output.txt', 'w') as file:
    for metric, value in metrics.items():
        file.write(f"{metrics}: {value}")